In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# Find the NovaBank project root.
candidates = [Path.cwd(), *Path.cwd().parents]

PROJECT_ROOT = next(
    (
        path for path in candidates
        if (path / "data" / "processed" /
            "credit_risk_cohort.csv").exists()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find the processed credit-risk dataset."
    )

DATA_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "credit_risk_cohort.csv"
)

REPORT_DIR = PROJECT_ROOT / "reports" / "credit_risk"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

credit = pd.read_csv(DATA_FILE)

print("Dataset shape:", credit.shape)
print("\nTarget distribution:")
print(credit["bad_outcome"].value_counts())
print("\nMissing target values:", credit["bad_outcome"].isna().sum())

In [ ]:
# Numeric borrower-profile features
numeric_candidates = [
    "loan_amnt",
    "annual_inc",
    "dti",
    "delinq_2yrs",
    "inq_last_6mths",
    "mths_since_last_delinq",
    "open_acc",
    "pub_rec",
    "revol_bal",
    "revol_util",
    "total_acc",
    "pub_rec_bankruptcies",
    "tax_liens",
    "acc_now_delinq",
    "collections_12_mths_ex_med",
    "credit_history_months",
]

# Categorical borrower-profile features
categorical_candidates = [
    "emp_length",
    "home_ownership",
    "verification_status",
    "purpose",
]

# Keep only features available and not entirely missing.
numeric_features = [
    col for col in numeric_candidates
    if col in credit.columns and credit[col].notna().any()
]

categorical_features = [
    col for col in categorical_candidates
    if col in credit.columns and credit[col].notna().any()
]

feature_columns = numeric_features + categorical_features

if not feature_columns:
    raise ValueError("No usable model features were found.")

# Explicitly exclude identifiers, outcomes and underwriting terms.
excluded_columns = [
    "record_id",
    "bad_outcome",
    "loan_status",
    "issue_date",
    "issue_year",
    "grade",
    "sub_grade",
    "term",
    "int_rate",
    "installment",
    "funded_amnt",
    "total_pymnt",
    "recoveries",
]

X = credit[feature_columns].copy()
y = credit["bad_outcome"].astype("int8")

# Convert numeric fields safely.
for col in numeric_features:
    X[col] = pd.to_numeric(X[col], errors="coerce")

# Convert missing categorical values to proper missing markers.
for col in categorical_features:
    X[col] = X[col].astype(object)
    X[col] = X[col].where(X[col].notna(), np.nan)

print("Selected numeric features:", numeric_features)
print("\nSelected categorical features:", categorical_features)
print("\nExcluded columns:", excluded_columns)
print("\nNumber of model features:", len(feature_columns))

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print("Training records:", len(X_train))
print("Test records:", len(X_test))

print(
    "\nTraining charged-off rate:",
    round(y_train.mean() * 100, 2),
    "%"
)

print(
    "Test charged-off rate:",
    round(y_test.mean() * 100, 2),
    "%"
)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print("Training records:", len(X_train))
print("Test records:", len(X_test))

print(
    "\nTraining charged-off rate:",
    round(y_train.mean() * 100, 2),
    "%"
)

print(
    "Test charged-off rate:",
    round(y_test.mean() * 100, 2),
    "%"
)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler

# Numeric data:
# 1. Fill missing values with the training-set median.
# 2. Add indicators for features that have missing values.
# 3. Scale using median and interquartile range.
numeric_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median", add_indicator=True)
    ),
    ("scaler", RobustScaler()),
])

# Categorical data:
# 1. Represent missing values explicitly.
# 2. Convert categories to indicator columns.
categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="constant", fill_value="Missing")
    ),
    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore",
            drop="first"
        )
    ),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

model = Pipeline([
    ("preprocess", preprocessor),
    (
        "logistic",
        LogisticRegression(
            max_iter=2000,
            solver="liblinear",
            random_state=42
        )
    ),
])

model.fit(X_train, y_train)

print("Logistic Regression training complete.")

In [ ]:
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    roc_auc_score,
)

test_probabilities = model.predict_proba(X_test)[:, 1]

# Constant-probability baseline: predict the training-set
# charged-off rate for every test loan.
baseline_probabilities = np.full(
    len(y_test),
    y_train.mean()
)

metrics = {
    "test_rows": len(y_test),
    "test_bad_rate_pct": round(y_test.mean() * 100, 2),
    "roc_auc": round(
        roc_auc_score(y_test, test_probabilities), 4
    ),
    "average_precision": round(
        average_precision_score(y_test, test_probabilities), 4
    ),
    "test_bad_rate_baseline_for_average_precision": round(
        y_test.mean(), 4
    ),
    "brier_score_model": round(
        brier_score_loss(y_test, test_probabilities), 4
    ),
    "brier_score_constant_baseline": round(
        brier_score_loss(y_test, baseline_probabilities), 4
    ),
}

for name, value in metrics.items():
    print(f"{name}: {value}")

In [ ]:
import math

scored_test = pd.DataFrame({
    "actual_bad_outcome": y_test.to_numpy(),
    "predicted_risk": test_probabilities,
})

scored_test = scored_test.sort_values(
    "predicted_risk",
    ascending=False
).reset_index(drop=True)

review_count = max(
    1,
    math.ceil(0.10 * len(scored_test))
)

top_review_group = scored_test.head(review_count)

total_bad_test = int(
    scored_test["actual_bad_outcome"].sum()
)

bad_found_in_review = int(
    top_review_group["actual_bad_outcome"].sum()
)

capture_rate = (
    bad_found_in_review / total_bad_test
    if total_bad_test else np.nan
)

precision_at_10_pct = (
    top_review_group["actual_bad_outcome"].mean()
)

print("Test loans:", len(scored_test))
print("Top 10% reviewed:", review_count)
print("Charged-off loans in test set:", total_bad_test)
print("Charged-off loans in top 10%:", bad_found_in_review)
print(f"Capture rate: {capture_rate:.2%}")
print(f"Precision in top 10%: {precision_at_10_pct:.2%}")

In [ ]:
coefficients = model.named_steps["logistic"].coef_[0]

feature_names = (
    model.named_steps["preprocess"].get_feature_names_out()
)

coefficient_table = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients,
})

coefficient_table["odds_ratio"] = np.exp(
    np.clip(coefficient_table["coefficient"], -20, 20)
)

print("Largest positive coefficients:")
display(
    coefficient_table
    .sort_values("coefficient", ascending=False)
    .head(15)
)

print("Largest negative coefficients:")
display(
    coefficient_table
    .sort_values("coefficient", ascending=True)
    .head(15)
)

coefficient_table.to_csv(
    REPORT_DIR / "logistic_regression_coefficients.csv",
    index=False,
)

In [ ]:
import json

metrics_path = REPORT_DIR / "logistic_regression_metrics.json"

with open(metrics_path, "w", encoding="utf-8") as file:
    json.dump(metrics, file, indent=4)

print("Metrics saved to:", metrics_path)
print(
    "Coefficients saved to:",
    REPORT_DIR / "logistic_regression_coefficients.csv"
)

In [ ]:
# 1. Overall model evaluation
print("========== MODEL EVALUATION ==========")

for name, value in metrics.items():
    print(f"{name}: {value}")


# 2. Investigation-capacity evaluation
print("\n========== TOP 10% INVESTIGATION ==========")
print(f"Test loans: {len(scored_test):,}")
print(f"Loans reviewed: {review_count:,}")
print(f"Charged-off loans in test set: {total_bad_test:,}")
print(f"Charged-off loans found in top 10%: {bad_found_in_review:,}")
print(f"Capture rate: {capture_rate:.2%}")
print(f"Precision in top 10%: {precision_at_10_pct:.2%}")


# 3. Largest positive and negative model coefficients
print("\n========== TOP POSITIVE COEFFICIENTS ==========")
display(
    coefficient_table
    .nlargest(10, "coefficient")[
        ["feature", "coefficient", "odds_ratio"]
    ]
)

print("\n========== TOP NEGATIVE COEFFICIENTS ==========")
display(
    coefficient_table
    .nsmallest(10, "coefficient")[
        ["feature", "coefficient", "odds_ratio"]
    ]
)


# 4. Coefficients for our key financial variables
print("\n========== KEY FINANCIAL VARIABLES ==========")

mask = coefficient_table["feature"].str.contains(
    r"annual_inc|dti|revol_util|loan_amnt|delinq_2yrs",
    case=False,
    regex=True
)

display(
    coefficient_table.loc[
        mask, ["feature", "coefficient", "odds_ratio"]
    ]
)

In [ ]:
import math
import pandas as pd

# Ensure highest predicted risk comes first.
ranked_test = (
    scored_test
    .sort_values("predicted_risk", ascending=False)
    .reset_index(drop=True)
)

total_test = len(ranked_test)
total_bad = int(ranked_test["actual_bad_outcome"].sum())
overall_bad_rate = ranked_test["actual_bad_outcome"].mean()

review_levels = [0.05, 0.10, 0.15, 0.20]
results = []

for review_pct in review_levels:

    # Number of accounts an analyst can review.
    n_review = max(
        1,
        math.ceil(total_test * review_pct)
    )

    top_group = ranked_test.head(n_review)

    bad_found = int(
        top_group["actual_bad_outcome"].sum()
    )

    capture_rate = (
        bad_found / total_bad
        if total_bad else 0
    )

    precision = (
        bad_found / n_review
    )

    results.append({
        "review_capacity_pct": round(review_pct * 100, 1),
        "loans_reviewed": n_review,
        "charged_off_loans_found": bad_found,
        "capture_rate_pct": round(capture_rate * 100, 2),
        "precision_pct": round(precision * 100, 2),
        "overall_bad_rate_pct": round(
            overall_bad_rate * 100, 2
        ),
        "precision_vs_baseline": round(
            precision / overall_bad_rate, 2
        ) if overall_bad_rate else None
    })

capacity_results = pd.DataFrame(results)

display(capacity_results)

capacity_results.to_csv(
    REPORT_DIR / "review_capacity_simulation.csv",
    index=False
)

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
)


# --------------------------------------------------
# 1. Load the prepared data
# --------------------------------------------------

candidates = [Path.cwd(), *Path.cwd().parents]

PROJECT_ROOT = next(
    (
        p for p in candidates
        if (p / "data" / "processed" /
            "credit_risk_cohort.csv").exists()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the processed NovaBank dataset."
    )

DATA_FILE = (
    PROJECT_ROOT
    / "data" / "processed" / "credit_risk_cohort.csv"
)

REPORT_DIR = PROJECT_ROOT / "reports" / "credit_risk"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

credit = pd.read_csv(DATA_FILE)

credit["issue_date"] = pd.to_datetime(
    credit["issue_date"],
    errors="coerce"
)

credit = credit.dropna(
    subset=["issue_date", "bad_outcome"]
).copy()

credit["bad_outcome"] = credit["bad_outcome"].astype(int)

# Work at month level so one issuance month cannot
# appear in both train and test.
credit["issue_month"] = credit["issue_date"].dt.to_period("M")

print("Rows with valid dates:", len(credit))
print(
    "Issuance period:",
    credit["issue_date"].min().date(),
    "to",
    credit["issue_date"].max().date()
)


# --------------------------------------------------
# 2. Select the same application-profile features
# --------------------------------------------------

numeric_candidates = [
    "loan_amnt",
    "annual_inc",
    "dti",
    "delinq_2yrs",
    "inq_last_6mths",
    "mths_since_last_delinq",
    "open_acc",
    "pub_rec",
    "revol_bal",
    "revol_util",
    "total_acc",
    "pub_rec_bankruptcies",
    "tax_liens",
    "acc_now_delinq",
    "collections_12_mths_ex_med",
    "credit_history_months",
]

categorical_candidates = [
    "emp_length",
    "home_ownership",
    "verification_status",
    "purpose",
]

numeric_features = [
    c for c in numeric_candidates
    if c in credit.columns and credit[c].notna().any()
]

categorical_features = [
    c for c in categorical_candidates
    if c in credit.columns and credit[c].notna().any()
]

feature_columns = numeric_features + categorical_features

if not feature_columns:
    raise ValueError("No eligible features were found.")

for c in numeric_features:
    credit[c] = pd.to_numeric(
        credit[c],
        errors="coerce"
    )


# --------------------------------------------------
# 3. Define chronological training and test periods
# --------------------------------------------------

months = sorted(credit["issue_month"].dropna().unique())

if len(months) < 5:
    raise ValueError(
        "Insufficient distinct issue months for temporal validation."
    )

split_position = int(len(months) * 0.80)
split_position = min(
    max(split_position, 1),
    len(months) - 1
)

cutoff_month = months[split_position]

train = credit[
    credit["issue_month"] < cutoff_month
].copy()

test = credit[
    credit["issue_month"] >= cutoff_month
].copy()

if train.empty or test.empty:
    raise ValueError(
        "Temporal split generated an empty training or test set."
    )

X_train_time = train[feature_columns].copy()
y_train_time = train["bad_outcome"]

X_test_time = test[feature_columns].copy()
y_test_time = test["bad_outcome"]

print("\nTemporal cutoff:", cutoff_month)
print("Training rows:", len(train))
print("Test rows:", len(test))

print(
    "Training dates:",
    train["issue_date"].min().date(),
    "to",
    train["issue_date"].max().date()
)

print(
    "Test dates:",
    test["issue_date"].min().date(),
    "to",
    test["issue_date"].max().date()
)

print(
    "Training charged-off rate:",
    round(y_train_time.mean() * 100, 2), "%"
)

print(
    "Test charged-off rate:",
    round(y_test_time.mean() * 100, 2), "%"
)


# --------------------------------------------------
# 4. Fit preprocessing using training data only
# --------------------------------------------------

numeric_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        )
    ),
    ("scaler", RobustScaler()),
])

categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="constant",
            fill_value="Missing"
        )
    ),
    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore",
            drop="first"
        )
    ),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    (
        "categorical",
        categorical_pipeline,
        categorical_features
    ),
])

temporal_model = Pipeline([
    ("preprocess", preprocessor),
    (
        "logistic",
        LogisticRegression(
            max_iter=2000,
            solver="liblinear",
            random_state=42
        )
    ),
])

temporal_model.fit(
    X_train_time,
    y_train_time
)

print("\nTemporal model training complete.")


# --------------------------------------------------
# 5. Evaluate later-issued loans
# --------------------------------------------------

temporal_probabilities = temporal_model.predict_proba(
    X_test_time
)[:, 1]

baseline_probabilities = np.full(
    len(y_test_time),
    y_train_time.mean()
)

temporal_metrics = {
    "training_rows": int(len(train)),
    "test_rows": int(len(test)),
    "training_bad_rate_pct": round(
        y_train_time.mean() * 100, 2
    ),
    "test_bad_rate_pct": round(
        y_test_time.mean() * 100, 2
    ),
    "roc_auc": round(
        roc_auc_score(
            y_test_time,
            temporal_probabilities
        ),
        4
    ),
    "average_precision": round(
        average_precision_score(
            y_test_time,
            temporal_probabilities
        ),
        4
    ),
    "test_prevalence_baseline": round(
        y_test_time.mean(),
        4
    ),
    "brier_score_model": round(
        brier_score_loss(
            y_test_time,
            temporal_probabilities
        ),
        4
    ),
    "brier_score_constant_baseline": round(
        brier_score_loss(
            y_test_time,
            baseline_probabilities
        ),
        4
    ),
}

print("\n===== TEMPORAL VALIDATION RESULTS =====")

for metric, value in temporal_metrics.items():
    print(f"{metric}: {value}")


# --------------------------------------------------
# 6. Save metrics and predictions
# --------------------------------------------------

with open(
    REPORT_DIR / "temporal_validation_metrics.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(temporal_metrics, file, indent=4)

temporal_predictions = pd.DataFrame({
    "record_id": test["record_id"].values,
    "issue_date": test["issue_date"].values,
    "actual_bad_outcome": y_test_time.values,
    "predicted_risk": temporal_probabilities,
})

temporal_predictions.to_csv(
    REPORT_DIR / "temporal_test_predictions.csv",
    index=False
)

print("\nMetrics and test predictions saved.")

In [ ]:
import math
import pandas as pd
import numpy as np

# Assemble the predictions from our temporal test set.
scored_temporal = pd.DataFrame({
    "record_id": test["record_id"].to_numpy(),
    "issue_date": test["issue_date"].to_numpy(),
    "actual_bad_outcome": y_test_time.to_numpy(),
    "predicted_risk": temporal_probabilities,
})

# Rank loans from highest to lowest predicted risk.
scored_temporal = (
    scored_temporal
    .sort_values("predicted_risk", ascending=False)
    .reset_index(drop=True)
)

total_loans = len(scored_temporal)
total_bad = int(scored_temporal["actual_bad_outcome"].sum())

results = []

for capacity in [0.05, 0.10, 0.15, 0.20]:
    number_to_review = max(
        1, math.ceil(total_loans * capacity)
    )

    reviewed = scored_temporal.head(number_to_review)

    bad_found = int(reviewed["actual_bad_outcome"].sum())

    results.append({
        "review_capacity_pct": capacity * 100,
        "loans_reviewed": number_to_review,
        "charged_off_loans_found": bad_found,
        "capture_rate_pct": round(
            bad_found / total_bad * 100, 2
        ) if total_bad else np.nan,
        "precision_pct": round(
            reviewed["actual_bad_outcome"].mean() * 100, 2
        ),
    })

capacity_results_temporal = pd.DataFrame(results)

display(capacity_results_temporal)

capacity_results_temporal.to_csv(
    REPORT_DIR / "temporal_review_capacity.csv",
    index=False,
)

In [ ]:
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt

# Group predictions into ten approximately equal-sized bins.
calibration_data = scored_temporal.copy()

calibration_data["risk_decile"] = pd.qcut(
    calibration_data["predicted_risk"],
    q=10,
    duplicates="drop",
)

calibration_table = (
    calibration_data
    .groupby("risk_decile", observed=True)
    .agg(
        loans=("actual_bad_outcome", "size"),
        mean_predicted_risk=("predicted_risk", "mean"),
        observed_bad_rate=("actual_bad_outcome", "mean"),
        minimum_predicted_risk=("predicted_risk", "min"),
        maximum_predicted_risk=("predicted_risk", "max"),
    )
    .reset_index()
)

calibration_table["mean_predicted_risk_pct"] = (
    calibration_table["mean_predicted_risk"] * 100
).round(2)

calibration_table["observed_bad_rate_pct"] = (
    calibration_table["observed_bad_rate"] * 100
).round(2)

display(calibration_table[[
    "risk_decile",
    "loans",
    "mean_predicted_risk_pct",
    "observed_bad_rate_pct",
]])

calibration_table.to_csv(
    REPORT_DIR / "temporal_calibration_by_decile.csv",
    index=False,
)

# Reliability diagram
fraction_positive, mean_predicted = calibration_curve(
    y_test_time,
    temporal_probabilities,
    n_bins=10,
    strategy="quantile",
)

fig, ax = plt.subplots(figsize=(7, 6))

ax.plot(
    mean_predicted,
    fraction_positive,
    marker="o",
    label="Logistic Regression",
)

ax.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Perfect calibration",
)

ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Observed charged-off rate")
ax.set_title("NovaBank — Temporal Calibration")
ax.legend()
fig.tight_layout()

fig.savefig(
    REPORT_DIR / "temporal_calibration.png",
    dpi=160,
    bbox_inches="tight",
)

plt.show()

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from xgboost import XGBClassifier

# Numeric variables:
# Fill missing values using training data only.
xgb_numeric_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        )
    )
])

# Categorical variables:
# Fill missing categories and convert them to numeric indicators.
xgb_categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="constant",
            fill_value="Missing"
        )
    ),
    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore",
            drop="first"
        )
    )
])

# Keep the same feature groups as the Logistic Regression model.
xgb_preprocessor = ColumnTransformer([
    ("numeric", xgb_numeric_pipeline, numeric_features),
    (
        "categorical",
        xgb_categorical_pipeline,
        categorical_features
    )
])

# A deliberately conservative first configuration.
xgb_classifier = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    n_estimators=400,
    max_depth=3,
    learning_rate=0.03,
    min_child_weight=5,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=5,
    tree_method="hist",
    random_state=42,
    n_jobs=4
)

xgb_model = Pipeline([
    ("preprocess", xgb_preprocessor),
    ("classifier", xgb_classifier)
])

# Train using earlier-issued loans only.
xgb_model.fit(X_train_time, y_train_time)

print("XGBoost training complete.")

In [ ]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss
)

# Predict on exactly the same later-issued loans.
xgb_temporal_probabilities = xgb_model.predict_proba(
    X_test_time
)[:, 1]

xgb_metrics = {
    "roc_auc": round(
        roc_auc_score(
            y_test_time,
            xgb_temporal_probabilities
        ), 4
    ),
    "average_precision": round(
        average_precision_score(
            y_test_time,
            xgb_temporal_probabilities
        ), 4
    ),
    "brier_score_model": round(
        brier_score_loss(
            y_test_time,
            xgb_temporal_probabilities
        ), 4
    ),
    "test_bad_rate_pct": round(
        y_test_time.mean() * 100, 2
    )
}

print("XGBoost temporal results:")

for metric, value in xgb_metrics.items():
    print(f"{metric}: {value}")

In [ ]:
comparison = pd.DataFrame([
    {
        "model": "Logistic Regression",
        "roc_auc": temporal_metrics["roc_auc"],
        "average_precision": temporal_metrics["average_precision"],
        "brier_score": temporal_metrics["brier_score_model"],
    },
    {
        "model": "XGBoost",
        "roc_auc": xgb_metrics["roc_auc"],
        "average_precision": xgb_metrics["average_precision"],
        "brier_score": xgb_metrics["brier_score_model"],
    }
])

display(comparison)

comparison.to_csv(
    REPORT_DIR / "temporal_model_comparison.csv",
    index=False
)

In [ ]:
import math
import numpy as np

def evaluate_review_capacity(
    actual,
    probabilities,
    model_name
):
    ranked = pd.DataFrame({
        "actual": np.asarray(actual),
        "probability": np.asarray(probabilities)
    }).sort_values(
        "probability",
        ascending=False
    ).reset_index(drop=True)

    total = len(ranked)
    total_bad = int(ranked["actual"].sum())

    rows = []

    for capacity in [0.05, 0.10, 0.15, 0.20]:
        n_review = max(
            1,
            math.ceil(total * capacity)
        )

        selected = ranked.head(n_review)
        bad_found = int(selected["actual"].sum())

        rows.append({
            "model": model_name,
            "review_capacity_pct": capacity * 100,
            "loans_reviewed": n_review,
            "charged_off_loans_found": bad_found,
            "capture_rate_pct": round(
                100 * bad_found / total_bad,
                2
            ) if total_bad else np.nan,
            "precision_pct": round(
                100 * selected["actual"].mean(),
                2
            )
        })

    return pd.DataFrame(rows)


review_comparison = pd.concat([
    evaluate_review_capacity(
        y_test_time,
        temporal_probabilities,
        "Logistic Regression"
    ),
    evaluate_review_capacity(
        y_test_time,
        xgb_temporal_probabilities,
        "XGBoost"
    )
], ignore_index=True)

display(
    review_comparison.sort_values(
        ["review_capacity_pct", "model"]
    )
)

review_comparison.to_csv(
    REPORT_DIR / "temporal_review_model_comparison.csv",
    index=False
)

In [ ]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
)

# Use the same temporal test observations for both models.
y_true = np.asarray(y_test_time, dtype=int)

lr_prob = np.asarray(temporal_probabilities)
xgb_prob = np.asarray(xgb_temporal_probabilities)

assert len(y_true) == len(lr_prob) == len(xgb_prob)

# Full-test-set differences: XGBoost minus Logistic Regression.
point_differences = {
    "roc_auc_delta": (
        roc_auc_score(y_true, xgb_prob)
        - roc_auc_score(y_true, lr_prob)
    ),
    "average_precision_delta": (
        average_precision_score(y_true, xgb_prob)
        - average_precision_score(y_true, lr_prob)
    ),
    "brier_score_delta": (
        brier_score_loss(y_true, xgb_prob)
        - brier_score_loss(y_true, lr_prob)
    ),
}

# Paired row-level bootstrap.
# This is an approximate uncertainty check, not a final
# guarantee of performance on future issuance periods.
rng = np.random.default_rng(42)
bootstrap_results = []

n = len(y_true)
n_bootstrap = 500

for iteration in range(n_bootstrap):

    indices = rng.integers(0, n, size=n)
    y_sample = y_true[indices]

    # AUC requires both classes in the resampled sample.
    if np.unique(y_sample).size < 2:
        continue

    lr_sample = lr_prob[indices]
    xgb_sample = xgb_prob[indices]

    bootstrap_results.append({
        "roc_auc_delta": (
            roc_auc_score(y_sample, xgb_sample)
            - roc_auc_score(y_sample, lr_sample)
        ),
        "average_precision_delta": (
            average_precision_score(y_sample, xgb_sample)
            - average_precision_score(y_sample, lr_sample)
        ),
        "brier_score_delta": (
            brier_score_loss(y_sample, xgb_sample)
            - brier_score_loss(y_sample, lr_sample)
        ),
    })

bootstrap_df = pd.DataFrame(bootstrap_results)

# Summarize the observed difference and its approximate
# 95% percentile-bootstrap interval.
summary_rows = []

for metric, point_delta in point_differences.items():

    lower = bootstrap_df[metric].quantile(0.025)
    upper = bootstrap_df[metric].quantile(0.975)

    summary_rows.append({
        "metric": metric,
        "xgboost_minus_logistic": point_delta,
        "bootstrap_95pct_lower": lower,
        "bootstrap_95pct_upper": upper,
    })

bootstrap_summary = pd.DataFrame(summary_rows)

display(
    bootstrap_summary.round(6)
)

bootstrap_summary.to_csv(
    REPORT_DIR / "paired_bootstrap_comparison.csv",
    index=False,
)

print("Bootstrap repetitions:", len(bootstrap_df))

In [ ]:
from pathlib import Path
import joblib

# Your actual NovaBank project folder
PROJECT_ROOT = Path(
    r"C:\Users\DELL\OneDrive\Desktop\NovaBank"
)

# Create the models folder if it doesn't exist
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Save the entire trained pipeline
MODEL_PATH = MODEL_DIR / "xgb_temporal_pipeline.joblib"

joblib.dump(xgb_model, MODEL_PATH)

print("Model saved successfully!")
print("Location:", MODEL_PATH)
print("File exists:", MODEL_PATH.exists())